# Create AERA Awards (American Educational Research Association)

Creates American Educational Research Association grants and fellowships from
AERA's own publications on www.aera.net:

1. **AERA Grants Program Research Grants and Dissertation Grants, 1991-2013**:
   AERA's "Funded Research Grants" and "Funded Dissertation Grants" pages (FileMaker
   exports: name, affiliation, project title, data set used, project dates).
2. **Newer recipients from AERA Highlights announcements** (AERA Newsroom):
   AERA-NSF Dissertation and Research Grantees 2020-2026, Minority Dissertation
   Fellows and their Travel Awardees 2015-2026, Fellowship Program on the Study of
   Deeper Learning cohorts 2019-2025. Announcement URLs were found through the
   Internet Archive's CDX index of aera.net/Newsroom/* and are read live.

**776 awards: 298 research grants, 336 dissertation grants, 75 Minority
Dissertation Fellowships, 41 Minority Dissertation Travel Awards, 26 Deeper
Learning fellowships; 100% title and PI, 99.7% institution, project dates for the
1991-2013 tables (66%), award year for everything** (local run 2026-10-01).
Gaps: no public recipient lists for the grants of 2014-2019 and 2025, or for the
Minority Dissertation Fellowship before 2015 (2014 announcement is prose only).

Scope (brief): fellowships, dissertation grants and research grants are all in
scope. **Flag:** the Minority Dissertation *Travel Awards* (`funding_type = travel`,
41 rows) fund the recipient's attendance at the AERA Annual Meeting, where they
present in the fellows' dissertation poster session; kept per the batch rule "when unsure, keep and flag".

**Money:** the AERA Grants Program is funded by NSF (DRL-0941014, DRL-1749275 ...)
and AERA makes the awards; the Deeper Learning fellowships are funded by the
Hewlett Foundation and awarded by AERA. All rows go to AERA as the awarding body
(citing works write "AERA Grants Program under NSF Grant #DRL-0941014").

**robots.txt:** aera.net disallows /Portals/ (the per-grant abstract pages) for
all agents, so abstracts are not fetched; no AI crawler is blocked.

**Prerequisites:**
- Run `scripts/local/aera_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/aera/aera_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** AERA publishes no citable grant number (citing
works cite the NSF grant). The 1991-2013 tables link each abstract as
`Abs-RG-00033204.html`: that record number is used (`AERA-RG-00033204`,
`AERA-DG-00032997`); other rows get `AERA-{RG|DG|MDF|MDF-TRAVEL|SDL}-{year}-{name-slug}`.
Collapses onto 0 citation shells.

**Description:** for the 1991-2013 grants, the data set(s) the project used.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320308275`
- display_name: American Educational Research Association
- ror_id / doi: from `openalex.funders.funders` (ror 00pmt4r91, doi 10.13039/100005165)

**Priority:** `598` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aera_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/aera/aera_projects.parquet`;

In [ ]:
%sql
SELECT program, COUNT(*) as n FROM openalex.awards.aera_raw GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
DESCRIBE openalex.awards.aera_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.aera_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320308275 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320308275;

## Step 2: Create AERA Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aera_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320308275  -- American Educational Research Association
),

base AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        g.*,
        named_struct(
            'given_name', NULLIF(TRIM(g.lead_given_name), ''),
            'family_name', NULLIF(TRIM(g.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) AS lead_struct,
        CASE WHEN NULLIF(TRIM(g.co_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(g.co_given_name), ''),
            'family_name', NULLIF(TRIM(g.co_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS co_struct,
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') AS sd,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') AS ed
    FROM openalex.awards.aera_raw g
    WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CASE WHEN NULLIF(TRIM(g.datasets), '') IS NOT NULL
         THEN CONCAT('Data set(s) used: ', TRIM(g.datasets)) END as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type,
    g.funder_scheme,
    'aera_funded_grants_fellowships' as provenance,
    g.sd as start_date,
    g.ed as end_date,
    COALESCE(YEAR(g.sd), TRY_CAST(g.year AS INT)) as start_year,
    YEAR(g.ed) as end_year,
    g.lead_struct as lead_investigator,
    g.co_struct as co_lead_investigator,
    CASE WHEN g.co_struct IS NOT NULL THEN ARRAY(g.lead_struct, g.co_struct) ELSE ARRAY(g.lead_struct) END as investigators,
    g.source_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM base g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aera_funded_grants_fellowships' AND priority = 598;

-- Insert into openalex_awards_raw with priority.
-- Priority 598: direct-from-funder ingest of AERA's own grant and fellowship lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    598 as priority
FROM openalex.awards.aera_awards
-- Kyle 2026-10-01: travel stipends are not research funding (Conquer Cancer/ASH rule): the 41
-- Minority Dissertation Fellowship Program travel awards are not shipped.
WHERE funding_type <> 'travel';

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids,
       (SELECT COUNT(*) FROM openalex.awards.aera_raw) as raw_rows
FROM openalex.awards.aera_awards;

In [ ]:
%sql
-- §2.1.1 shape check: AERA record number or the synthetic AERA-{program}-{year}-{slug} form.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^AERA-(RG|DG)-[0-9]{8}$' THEN 1 ELSE 0 END) as record_numbers,
    SUM(CASE WHEN funder_award_id RLIKE '^AERA-(RG|DG|MDF|MDF-TRAVEL|SDL)-[0-9]{4}-[a-z0-9-]+$' THEN 1 ELSE 0 END) as synthetic,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^AERA-(RG|DG)-[0-9]{8}$'
              AND funder_award_id NOT RLIKE '^AERA-(RG|DG|MDF|MDF-TRAVEL|SDL)-[0-9]{4}-[a-z0-9-]+$' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.aera_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_date, end_date, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.aera_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, funding_type, COUNT(*) as cnt FROM openalex.awards.aera_awards GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.aera_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.aera_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage: AERA publishes program-level amounts only (e.g. dissertation
-- grants "$25,000 for one year"), not per award, so amount is NULL by design.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.aera_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aera_funded_grants_fellowships'
GROUP BY provenance, priority;